# 12 — Application Prompt Training Data

Turns real-world ARO applications into instruction-tuning pairs.

Each application under `/Users/kris/Projects/ARO-Application` that contains a
`PROMPT.md` file (the natural-language requirement used to build that app) is
converted into training data:

1. Read `PROMPT.md` as the user instruction.
2. Collect all source files from the same application (`.aro`, `.screen`, `openapi.yaml`, etc.).
3. Produce one **ground-truth** pair: `instruction = PROMPT.md`, `output = all source files`.
4. Use the local LLM to generate **4 alternative prompts** (shorter, less precise,
   description-only) — each paired with the same code output.
5. Append all pairs to `knowledge_pairs.jsonl`.

**Input:**  `/Users/kris/Projects/ARO/ARO-Application/*/PROMPT.md` — one file per application
            Source files: `.aro`, `.yaml`, `.yml`, `.screen`, `.json`, `.toml`

**Output:** `application_prompt_pairs.jsonl` — 15 pairs (3 apps × 5 prompts each)
            Appended to `../data/02_knowledge/knowledge_pairs.jsonl`

**Applications processed (last run):**

| Application | Source files | Pairs |
|-------------|-------------|------:|
| StatusPost | 6 | 5 |
| Sumup | 3 | 5 |
| SystemLoadMonitor | 3 | 5 |
| **Total** | | **15** |

## Setup

In [1]:
import sys, importlib, json, re, textwrap
from pathlib import Path

_cfg_dir = Path('.').resolve()
if str(_cfg_dir) not in sys.path:
    sys.path.insert(0, str(_cfg_dir))
import config; importlib.reload(config); from config import *

OUTPUT_FILE = SCRIPT_DIR / 'application_prompt_pairs.jsonl'

# Source file extensions to include in the output (excluding READMEs and hints)
CODE_EXTENSIONS = {'.aro', '.yaml', '.yml', '.screen', '.json', '.toml'}
# Filenames to always include even without a matching extension
ALWAYS_INCLUDE  = {'openapi.yaml', 'openapi.yml', 'plugin.yaml'}
# Directories to skip
SKIP_DIRS = {'Archive', '.git', '__pycache__', 'node_modules', 'output', 'deployment'}

SYSTEM = """You are an expert ARO application architect.
ARO (Action Result Object) is a DSL for expressing business logic as natural-language statements.
ARO applications are directories of .aro files, optionally with an openapi.yaml for HTTP API servers.
Feature sets are named after business activities and triggered by events (HTTP routes, domain events, file events).
You help developers articulate what ARO applications should do in plain business language."""

print('Setup complete.')
print(f'  ARO-Application root : {ARO_APPLICATION_ROOT}')
print(f'  Output file          : {OUTPUT_FILE}')

TRAIN_ON_BASE=True → using base model: mlx-community/Qwen3-Coder-30B-A3B-Instruct-4bit
TRAIN_ON_BASE=True → using base model: mlx-community/Qwen3-Coder-30B-A3B-Instruct-4bit
Setup complete.
  ARO-Application root : /Users/kris/Projects/ARO/ARO-Application
  Output file          : /Users/kris/Projects/ARO/ARO-Train/Train/script/application_prompt_pairs.jsonl


## Load model

In [2]:
model, tokenizer, _load_fn, mlx_generate, make_sampler = load_model()
print('Model ready.')

/Users/kris/Projects/ARO/ARO-Train/Train/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading mlx-community/Qwen3-Coder-30B-A3B-Instruct-4bit with warm-start adapter...


Fetching 15 files: 100%|██████████| 15/15 [00:00<00:00, 59521.82it/s]


  Adapter: /Users/kris/Projects/ARO/ARO-Train/Train/data/adapters/warm_start
Model ready.
Model ready.


## Helpers

In [3]:
def chat(messages: list[dict], max_tokens: int = 800, temperature: float = 0.7) -> str:
    """Run the loaded model on a message list and return the response string."""
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    sampler = make_sampler(temp=temperature)
    return mlx_generate(model, tokenizer, prompt=prompt, max_tokens=max_tokens, sampler=sampler)


def collect_app_code(app_dir: Path) -> str:
    """
    Walk app_dir and collect all source files into a single formatted string.
    Each file is preceded by a header showing its relative path.
    Files are ordered: openapi.yaml first, then .aro files alphabetically,
    then template files, then anything else.
    """
    files: list[Path] = []
    for path in sorted(app_dir.rglob('*')):
        if not path.is_file():
            continue
        # Skip unwanted directories
        if any(part in SKIP_DIRS for part in path.parts):
            continue
        # Skip the plan.md itself and README/test files
        if path.name in ('plan.md', 'README.md', 'test.hint'):
            continue
        if path.suffix.lower() in CODE_EXTENSIONS or path.name in ALWAYS_INCLUDE:
            files.append(path)

    # Sort: openapi first, .aro next, then templates, then rest
    def sort_key(p: Path):
        if p.name.startswith('openapi'):
            return (0, p.name)
        if p.suffix == '.aro':
            return (1, p.name)
        if p.suffix == '.screen':
            return (2, p.name)
        return (3, p.name)

    files.sort(key=sort_key)

    parts = []
    for f in files:
        rel = f.relative_to(app_dir)
        content = f.read_text(errors='replace').rstrip()
        parts.append(f'# {rel}\n{content}')

    return '\n\n'.join(parts)


def generate_alternative_prompts(original_prompt: str, n: int = 4) -> list[str]:
    """
    Ask the LLM to produce n alternative prompts that:
    - Are shorter and less precise than the original
    - Describe only WHAT the application does (not HOW)
    - Contain no implementation details (no mention of ARO actions, feature sets,
      specific file structures, repository names, port numbers, etc.)
    - Sound like a natural request from a developer who wants an application built
    Returns a list of prompt strings (best-effort: fewer than n if parsing fails).
    """
    messages = [
        {'role': 'system', 'content': (
            'You are a technical writer specialising in software requirements. '
            'Your task is to produce shorter, vaguer paraphrases of a requirement '
            'that keep the high-level intent but strip all implementation details.'
        )},
        {'role': 'user', 'content': textwrap.dedent(f"""
            Below is the original requirement used to build an ARO application.
            Produce exactly {n} alternative phrasings of this requirement.

            Rules:
            - Each alternative must describe only WHAT the application does, not HOW.
            - Strip all implementation details: no mention of ARO, feature sets, actions,
              specific file names, port numbers, repository names, template names, or
              any ARO-specific syntax.
            - Each alternative should be shorter and less precise than the original.
            - Write as a developer making a request ("Build me...", "I need...", "Create...").
            - Output ONLY the {n} alternatives, one per line, numbered 1. 2. 3. 4.
              Do not add any other text.

            ORIGINAL:
            {original_prompt.strip()}
        """).strip()},
    ]

    response = chat(messages, max_tokens=600, temperature=0.8)

    # Parse numbered lines
    alternatives = []
    for line in response.splitlines():
        m = re.match(r'^\s*\d+[.)\s]+(.+)', line)
        if m:
            text = m.group(1).strip()
            if text:
                alternatives.append(text)

    return alternatives[:n]


def make_pair(instruction: str, code_output: str, source: str, task_type: str = 'code_generation') -> dict:
    """Build a standard training pair dict."""
    return {
        'messages': [
            {'role': 'system',    'content': SYSTEM},
            {'role': 'user',      'content': instruction},
            {'role': 'assistant', 'content': code_output},
        ],
        'task_type': task_type,
        'source': source,
    }


print('Helpers loaded.')

Helpers loaded.


## Resume support

In [4]:
# Track which app directories have already been processed so the cell is re-runnable.
done_apps: set[str] = set()
if OUTPUT_FILE.exists():
    with OUTPUT_FILE.open() as f:
        for line in f:
            try:
                rec = json.loads(line)
                src = rec.get('source', '')
                # source format: "app_name/prompt" or "app_name/alt_N"
                app_name = src.split('/')[0]
                done_apps.add(app_name)
            except Exception:
                pass

print(f'Already processed: {sorted(done_apps) or "(none)"}')

Already processed: ['Buildflow', 'Crawler', 'FediTerm', 'MastoStream', 'StatusPost', 'Sumup', 'SystemLoadMonitor', 'mm', 'mostrecentfile', 'rulpz', 'sf']


## Main loop — collect prompts, code, and generate alternatives

In [5]:
all_pairs: list[dict] = []
stats: dict[str, dict] = {}

prompt_files = sorted(ARO_APPLICATION_ROOT.rglob('plan.md'))
print(f'Found {len(prompt_files)} plan.md file(s) under {ARO_APPLICATION_ROOT}\n')

for prompt_path in prompt_files:
    app_dir  = prompt_path.parent
    app_name = app_dir.name

    if app_name in done_apps:
        print(f'  {app_name:<25}  skipped (already in output file)')
        continue

    # ── 1. Read the prompt ────────────────────────────────────────────────────
    original_prompt = prompt_path.read_text().strip()
    if not original_prompt:
        print(f'  {app_name:<25}  skipped (empty plan.md)')
        continue

    # ── 2. Collect all application source code ────────────────────────────────
    code_output = collect_app_code(app_dir)
    if not code_output:
        print(f'  {app_name:<25}  skipped (no source files found)')
        continue

    pairs_for_app: list[dict] = []

    # ── 3. Ground-truth pair (original prompt → full code) ────────────────────
    pairs_for_app.append(make_pair(
        instruction = original_prompt,
        code_output = code_output,
        source      = f'{app_name}/prompt',
    ))

    # ── 4. Generate 4 shorter alternative prompts via LLM ────────────────────
    print(f'  {app_name:<25}  generating alternatives...', end='', flush=True)
    alternatives = generate_alternative_prompts(original_prompt, n=9)
    print(f'  got {len(alternatives)}')

    for i, alt in enumerate(alternatives, 1):
        pairs_for_app.append(make_pair(
            instruction = alt,
            code_output = code_output,
            source      = f'{app_name}/alt_{i}',
            task_type   = 'code_generation_paraphrase',
        ))

    # ── 5. Save to JSONL ──────────────────────────────────────────────────────
    with OUTPUT_FILE.open('a') as f:
        for p in pairs_for_app:
            f.write(json.dumps(p) + '\n')

    all_pairs.extend(pairs_for_app)
    done_apps.add(app_name)
    stats[app_name] = {
        'files': len([l for l in code_output.split('\n') if l.startswith('# ')]),
        'pairs': len(pairs_for_app),
        'alternatives': len(alternatives),
        'prompt_len': len(original_prompt),
    }
    print(f'  {app_name:<25}  {stats[app_name]["files"]} files  →  {len(pairs_for_app)} pairs')

print(f'\nTotal new pairs: {len(all_pairs)}')
print(f'Saved → {OUTPUT_FILE}')

Found 11 plan.md file(s) under /Users/kris/Projects/ARO/ARO-Application

  Buildflow                  skipped (already in output file)
  Crawler                    skipped (already in output file)
  FediTerm                   skipped (already in output file)
  MastoStream                skipped (already in output file)
  StatusPost                 skipped (already in output file)
  Sumup                      skipped (already in output file)
  SystemLoadMonitor          skipped (already in output file)
  mm                         skipped (already in output file)
  mostrecentfile             skipped (already in output file)
  rulpz                      skipped (already in output file)
  sf                         skipped (already in output file)

Total new pairs: 0
Saved → /Users/kris/Projects/ARO/ARO-Train/Train/script/application_prompt_pairs.jsonl


## Inspect samples

In [6]:
import random

# Reload all pairs from disk for inspection (handles re-run case)
_all: list[dict] = []
if OUTPUT_FILE.exists():
    with OUTPUT_FILE.open() as f:
        for line in f:
            if line.strip():
                _all.append(json.loads(line))

originals  = [p for p in _all if p.get('source', '').endswith('/prompt')]
paraphrases = [p for p in _all if '/alt_' in p.get('source', '')]

print(f'Total pairs on disk : {len(_all)}')
print(f'  Original prompts  : {len(originals)}')
print(f'  Paraphrases       : {len(paraphrases)}')
print()

# Show one original + its alternatives
if originals:
    sample_orig = random.choice(originals)
    app_src = sample_orig.get('source','').replace('/prompt', '')
    _hr = '─' * 72
    print(_hr)
    print(f'APP: {app_src}')
    print(_hr)
    print('ORIGINAL PROMPT:')
    print(sample_orig['messages'][1]['content'][:400])
    print()
    print('CODE OUTPUT (first 600 chars):')
    print(sample_orig['messages'][2]['content'][:600])
    print('...')
    print()

    alts = [p for p in _all if p.get('source', '').startswith(app_src + '/alt_')]
    if alts:
        print('ALTERNATIVE PROMPTS:')
        for i, alt_pair in enumerate(alts, 1):
            print(f'  {i}. {alt_pair["messages"][1]["content"]}')

Total pairs on disk : 95
  Original prompts  : 11
  Paraphrases       : 84

────────────────────────────────────────────────────────────────────────
APP: MastoStream
────────────────────────────────────────────────────────────────────────
ORIGINAL PROMPT:
# Build a Mastodon timeline viewer

Build a terminal application that connects to a Mastodon instance, displays the home timeline, and streams live updates. The user should be able to scroll through posts with arrow keys, compose new posts, and quit. Read instance URL and access token from a config file. Keep it simpler than a full client — no reply functionality needed.

CODE OUTPUT (first 600 chars):
# openapi.yaml
openapi: 3.0.3
info:
  title: MastoStream
  version: 2.0.0
  description: Mastodon terminal client — no HTTP server, schemas only
paths: {}
components:
  schemas:
    UIState:
      type: object
      properties:
        key:          { type: string }
        mode:         { type: string, enum: [timeline, compose, setup] 

## Append to knowledge_pairs.jsonl

In [7]:
PAIRS_FILE.parent.mkdir(parents=True, exist_ok=True)

clean_notebook_pairs('NB11')

pairs_to_save = []
if OUTPUT_FILE.exists():
    with OUTPUT_FILE.open() as f:
        for line in f:
            if not line.strip():
                continue
            try:
                rec = json.loads(line)
                rec['notebook'] = 'NB11'
                pairs_to_save.append(rec)
            except Exception:
                pass

new_count = save_notebook_pairs('NB11', pairs_to_save)

print(f'Appended {new_count} new pairs to {PAIRS_FILE}')

[NB11] Cleaned 460 pairs from previous run
Appended 95 new pairs to /Users/kris/Projects/ARO/ARO-Train/Train/data/02_knowledge/knowledge_pairs.jsonl


## Summary

In [8]:
print('═' * 60)
print('Application Prompt Training Data — Summary')
print('═' * 60)

if stats:
    print(f'\n{"Application":<25}  {"Files":>5}  {"Alts":>4}  {"Pairs":>5}')
    print('─' * 50)
    for name, s in sorted(stats.items()):
        print(f'{name:<25}  {s["files"]:>5}  {s["alternatives"]:>4}  {s["pairs"]:>5}')
    print('─' * 50)
    total = sum(s['pairs'] for s in stats.values())
    print(f'{"TOTAL":<25}  {"":>5}  {"":>4}  {total:>5}')
else:
    # Recompute from disk
    _counts: dict[str, int] = {}
    if OUTPUT_FILE.exists():
        with OUTPUT_FILE.open() as f:
            for line in f:
                if line.strip():
                    try:
                        src = json.loads(line).get('source', '')
                        app = src.split('/')[0]
                        _counts[app] = _counts.get(app, 0) + 1
                    except Exception:
                        pass
    for app, n in sorted(_counts.items()):
        print(f'  {app:<25}  {n} pairs')
    print(f'\nTotal : {sum(_counts.values())} pairs')

print(f'\nOutput : {OUTPUT_FILE}')
print(f'Merged : {PAIRS_FILE}')

════════════════════════════════════════════════════════════
Application Prompt Training Data — Summary
════════════════════════════════════════════════════════════
  Buildflow                  10 pairs
  Crawler                    10 pairs
  FediTerm                   10 pairs
  MastoStream                10 pairs
  StatusPost                 5 pairs
  Sumup                      5 pairs
  SystemLoadMonitor          5 pairs
  mm                         10 pairs
  mostrecentfile             10 pairs
  rulpz                      10 pairs
  sf                         10 pairs

Total : 95 pairs

Output : /Users/kris/Projects/ARO/ARO-Train/Train/script/application_prompt_pairs.jsonl
Merged : /Users/kris/Projects/ARO/ARO-Train/Train/data/02_knowledge/knowledge_pairs.jsonl


In [9]:
import csv
from pathlib import Path
from datetime import date as _date

_run_dir = Path('.') / 'run' / _date.today().isoformat()
_run_dir.mkdir(parents=True, exist_ok=True)
_csv_out = _run_dir / '11_application_prompts.csv'

# Reload from disk to be self-contained
_apps_csv: dict[str, dict] = {}
if OUTPUT_FILE.exists():
    with OUTPUT_FILE.open() as _f:
        for _line in _f:
            if not _line.strip():
                continue
            try:
                _rec = json.loads(_line)
                _src = _rec.get('source', '')
                _app = _src.split('/')[0]
                _kind = 'original' if _src.endswith('/prompt') else 'paraphrase'
                if _app not in _apps_csv:
                    _apps_csv[_app] = {'original': 0, 'paraphrase': 0}
                _apps_csv[_app][_kind] += 1
            except Exception:
                pass

with open(_csv_out, 'w', newline='') as _cf:
    w = csv.writer(_cf)
    w.writerow(['app_name', 'original_prompts', 'paraphrases', 'total'])
    for _name in sorted(_apps_csv):
        o = _apps_csv[_name]['original']
        p = _apps_csv[_name]['paraphrase']
        w.writerow([_name, o, p, o + p])

print(f'CSV saved: {_csv_out}  ({len(_apps_csv)} rows)')

CSV saved: run/2026-04-30/11_application_prompts.csv  (11 rows)


In [10]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
plt.style.use('default')
plt.rcParams.update({
    'text.color': '#222222',
    'axes.labelcolor': '#222222',
    'xtick.color': '#333333',
    'ytick.color': '#333333',
    'axes.titlecolor': '#111111',
    'legend.edgecolor': '#cccccc',
    'legend.facecolor': 'white',
    'legend.framealpha': 1.0,
    'figure.facecolor': '#fafafa',
    'axes.facecolor': '#f9f9f9',
    'savefig.facecolor': '#fafafa',
    'savefig.bbox': 'tight',
    'savefig.dpi': 150,
})
from pathlib import Path
import json
import numpy as np
from datetime import date as _date

_run_dir = Path('.') / 'run' / _date.today().isoformat()
_run_dir.mkdir(parents=True, exist_ok=True)
_out = _run_dir / '11_application_prompts.png'

# ── Load pairs from disk ─────────────────────────────────────────────────────
_apps: dict[str, dict] = {}
if OUTPUT_FILE.exists():
    with OUTPUT_FILE.open() as _f:
        for _line in _f:
            if not _line.strip():
                continue
            try:
                _rec  = json.loads(_line)
                _src  = _rec.get('source', '')
                _app  = _src.split('/')[0]
                _kind = 'original' if _src.endswith('/prompt') else 'paraphrase'
                if _app not in _apps:
                    _apps[_app] = {'original': 0, 'paraphrase': 0}
                _apps[_app][_kind] += 1
            except Exception:
                pass

if not _apps:
    print('No pairs found — run the main loop cell first.')
else:
    _labels = sorted(_apps.keys())
    _orig   = [_apps[a]['original']   for a in _labels]
    _para   = [_apps[a]['paraphrase']  for a in _labels]
    _total  = [o + p for o, p in zip(_orig, _para)]
    _total_pairs = sum(_total)

    y   = np.arange(len(_labels))
    h   = 0.45
    fig, ax = plt.subplots(figsize=(10, max(3, len(_labels) * 0.9 + 1.5)))

    b1 = ax.barh(y, _orig, h, label='Original prompt',  color='#3498db', edgecolor='white')
    b2 = ax.barh(y, _para, h, left=_orig,               label='Paraphrases (×4)', color='#2ecc71', edgecolor='white')

    ax.set_yticks(y)
    ax.set_yticklabels(_labels, fontsize=10)
    ax.set_xlabel('Training pairs')
    ax.set_title(
        f'Application Prompts — {_total_pairs:,} pairs from {len(_labels)} application(s)',
        fontsize=13, fontweight='bold'
    )
    ax.legend(fontsize=10, loc='lower right')
    ax.grid(axis='x', alpha=0.3)

    # Annotate total per app at end of bar
    for i, tot in enumerate(_total):
        ax.text(tot + 0.05, i, str(tot), va='center', ha='left', fontsize=9, color='#333')

    fig.tight_layout()
    fig.savefig(_out)
    plt.close(fig)
    print(f'Saved: {_out}')

Saved: run/2026-04-30/11_application_prompts.png
